Test with data before building the EM, pruning, etc.

Create a simple vocabulary for training

In [ ]:
vocab = {
    "play": 0.30,
    "ing": 0.20,
    "playing": 0.50,
    "pla": 0.05,
    "ying": 0.05,
    "p": 0.01,
    "l": 0.01,
    "a": 0.01,
    "y": 0.01,
    "i": 0.01,
    "n": 0.01,
    "g": 0.01,
}

Generate all valid token matches

In [ ]:
def find_matches(text, vocab):
    matches = {}
    for start in range(len(text)):
        matches[start] = []
        for token in vocab:
            if text.startswith(token, start):
                matches[start].append(token)
    return matches

In [ ]:
text = "playing"

matches = find_matches(text, vocab)
for pos, tokens in matches.items():
    if tokens:
        print(pos, tokens)

Convert probabilities to scores

P(segmentation) = P(token1) * P(token2) *....

Multiplication is not ideal, we take logs:

log(P1 x P2) = log(P1) + log(P2)

In [ ]:
import math

In [ ]:
log_vocab = {
    token: math.log(prob)
    for token, prob in vocab.items()
}

VIterbi Dynamic programming

In [ ]:
def viterbi(text, vocab):
    import math
    n = len(text)

    best_score = [-float("inf")] * (n + 1)
    best_prev = [-1] * (n + 1)
    best_token = [""] * (n + 1)
    best_score[0] = 0

    for pos in range(n):
        if best_score[pos] == -float("inf"):
            continue
        for token, prob in vocab.items():
            if text.startswith(token, pos):
                next_pos = pos + len(token)
                score = best_score[pos] + math.log(prob)
                if score > best_score[next_pos]:
                    best_score[next_pos] = score
                    best_prev[next_pos] = pos
                    best_token[next_pos] = token

    return best_score, best_prev, best_token

Recover the best path

In [ ]:
def backtrack(text, best_prev, best_token):
    pos = len(text)
    tokens = []
    while pos > 0:
        tokens.append(best_token[pos])
        pos = best_prev[pos]
    return list(reversed(tokens))

In [ ]:
scores, prev, tok = viterbi("playing", vocab)
tokens = backtrack("playing", prev, tok)

print(tokens)

# Moving on to EM

In [ ]:
from collections import Counter

def e_step(corpus, vocab):
    counts = Counter()
    for text in corpus:
        scores, prev, tok = viterbi(text, vocab)
        tokens = backtrack(text, prev, tok)
        counts.update(tokens)
    return counts

In [ ]:
corpus = [
    "playing",
    "played",
    "player",
    "playing",
    "playing",
]

counts = e_step(corpus, vocab)
print(counts)

Idea: If a token was used frequently, it's probability needs to increase.

P(token) = token_count/total_token_count

In [ ]:
def m_step(counts):
    total = sum(counts.values())
    new_vocab = {}
    for token, count in counts.items():
        new_vocab[token] = count / total
    return new_vocab

In [ ]:
new_vocab = m_step(counts)

for token, prob in sorted(
    new_vocab.items(),
    key=lambda x: x[1],
    reverse=True,
):
    print(token, round(prob, 3))

One EM Iteration

In [ ]:
def em_iteration(corpus, vocab):
    counts = e_step(corpus, vocab)
    vocab = m_step(counts)
    return vocab

In [ ]:
vocab = em_iteration(corpus, vocab)
print(vocab)